In [1]:
%pip install uv --quiet
!uv pip install pandas numpy plotly matplotlib
!uv sync

Note: you may need to restart the kernel to use updated packages.


c:\Users\cbutt\OneDrive\Desktop\DataScience\Project\Modern-Store-of-Value\.venv\Scripts\python.exe: No module named pip
Using Python 3.12.3 environment at: c:\Users\cbutt\OneDrive\Desktop\DataScience\Project\Modern-Store-of-Value\.venv
Checked 4 packages in 464ms
Resolved 147 packages in 2ms
Uninstalled 9 packages in 215ms
 - choreographer==1.2.1
 - iniconfig==2.3.0
 - kaleido==1.2.0
 - logistro==2.0.1
 - orjson==3.11.8
 - pluggy==1.6.0
 - pytest==9.0.3
 - pytest-timeout==2.4.0
 - simplejson==3.20.2


## Combine csvs

In [24]:
import pandas as pd
import os

BASE_DIR = os.path.abspath("../data")

FILES = {
    "sortino":             os.path.join(BASE_DIR, "sortino_metric.csv"),
    "calmar":              os.path.join(BASE_DIR, "calmar_metrics.csv"),
    "inflation":           os.path.join(BASE_DIR, "inflation_metrics.csv"),
    "market_independence": os.path.join(BASE_DIR, "market_independence_metrics.csv"),
    "crisis":              os.path.join(BASE_DIR, "crisis_metrics.csv"),
}

def load(name: str, path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "Asset_Name" in df.columns:
        df = df.rename(columns={"Asset_Name": "Ticker"})

    rename_map = {
        col: f"{name}__{col}"
        for col in df.columns
        if col != "Ticker"
    }
    return df.rename(columns=rename_map)


dfs = [load(name, path) for name, path in FILES.items()]

merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on="Ticker", how="outer")

merged = merged.sort_values("Ticker").reset_index(drop=True)

# ── Consolidate Category columns into one ─────────────────────────────────────
category_cols = [col for col in merged.columns if col.endswith("__Category")]
merged["Category"] = merged[category_cols].bfill(axis=1).iloc[:, 0]
merged = merged.drop(columns=category_cols)

# Move Category to the front, right after Ticker
cols = ["Ticker", "Category"] + [c for c in merged.columns if c not in ("Ticker", "Category")]
merged = merged[cols]

# ── Save ──────────────────────────────────────────────────────────────────────
output_path = os.path.join(BASE_DIR, "merged_metrics.csv")
merged.to_csv(output_path, index=False)

print(f"✅  Merged {len(merged)} tickers × {len(merged.columns)} columns")
print(f"    Saved → {output_path}\n")
print(merged.to_string(max_rows=10))

✅  Merged 23 tickers × 16 columns
    Saved → c:\Users\cbutt\OneDrive\Desktop\DataScience\Project\Modern-Store-of-Value\data\merged_metrics.csv

   Ticker                      Category  sortino__Sortino_Ratio  sortino__Normalized_Sortino_Score_1_10  calmar__Raw_Calmar_Ratio  calmar__Normalized_Calmar_Score_1_10  inflation__Raw_Real_Return_%  inflation__Normalized_Inflation_Score_1_10  market_independence__Raw_Correlation  market_independence__Raw_Beta  market_independence__Correlation_MinMax  market_independence__Beta_MinMax  market_independence__Correlation_Percentile  market_independence__Beta_Percentile  crisis__Raw_Mean_Stress_Score  crisis__Normalized_Crisis_Score_1_10
0    AAPL             Individual Stocks                  1.0685                                    6.73                  0.532745                                  7.55                     72.214169                                        7.95                                0.6545                         1.0348       

## Calculate using Normalized metrics

In [22]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load master CSV ───────────────────────────────────────────────────────
BASE_DIR = os.path.abspath("../data")
merged = pd.read_csv(os.path.join(BASE_DIR, "merged_metrics.csv"))

# ── Define the five score columns and their display labels ────────────────
SCORE_COLS = [
    "sortino__Normalized_Sortino_Score_1_10",
    "calmar__Normalized_Calmar_Score_1_10",
    "inflation__Normalized_Inflation_Score_1_10",
    "market_independence__Correlation_Percentile",
    "crisis__Normalized_Crisis_Score_1_10",
]

LABELS = [
    "Sortino\n(Risk-Adj.)",
    "Calmar\n(Drawdown)",
    "Inflation\nAdj. Return",
    "Market\nIndependence",
    "Crisis\nPerformance",
]

# ── Calculate overall score = mean of all five metrics per ticker ─────────
merged["Overall_Score"] = merged[SCORE_COLS].mean(axis=1).round(2)
merged = merged.sort_values("Overall_Score", ascending=False).reset_index(drop=True)
merged["Rank"] = merged.index + 1

print("=== LEADERBOARD ===")
print(merged[["Rank", "Ticker", "Category", "Overall_Score"]].to_string(index=False))

=== LEADERBOARD ===
 Rank Ticker                     Category  Overall_Score
    1    GLD      Commodity ETFs (Metals)           9.02
    2    WMT            Individual Stocks           8.69
    3    DBA Commodity ETFs (Agriculture)           8.36
    4    SLV      Commodity ETFs (Metals)           7.47
    5   NVDA            Individual Stocks           7.06
    6    XLU                  Sector ETFs           6.97
    7   IBIT                  Crypto ETFs           6.81
    8   PPLT      Commodity ETFs (Metals)           6.64
    9   AAPL            Individual Stocks           6.15
   10    JNJ            Individual Stocks           5.99
   11    AMD            Individual Stocks           5.83
   12    VTI            Broad Market ETFs           5.81
   13    SPY            Broad Market ETFs           5.75
   14   SOYB Commodity ETFs (Agriculture)           5.26
   15   MSFT            Individual Stocks           4.76
   16   WEAT Commodity ETFs (Agriculture)           3.95
   17    LO

## Calculate using raw metric values (Normalized 1-10)

In [27]:
# ── Define direction of each metric ───────────────────────────────────────
# True = Higher is better (e.g., Returns)
# False = Lower is better (e.g., Stress/Correlation)
METRIC_DIRECTIONS = {
    "sortino__Sortino_Ratio": True,
    "calmar__Raw_Calmar_Ratio": True,
    "inflation__Raw_Real_Return_%": True,
    "market_independence__Raw_Correlation": False,
    "crisis__Raw_Mean_Stress_Score": False
}

SCORE_COLS = list(METRIC_DIRECTIONS.keys())
normalized_col_names = []

# ── Perform Min-Max Normalization (1-10 Scale) ───────────────────────────
for col, higher_is_better in METRIC_DIRECTIONS.items():
    norm_name = f"norm_{col}"
    normalized_col_names.append(norm_name)
    
    col_min = merged[col].min()
    col_max = merged[col].max()
    
    if higher_is_better:
        # Standard: 10 is the Max, 1 is the Min
        merged[norm_name] = 1 + 9 * (merged[col] - col_min) / (col_max - col_min)
    else:
        # Inverted: 10 is the Min (best), 1 is the Max (worst)
        merged[norm_name] = 1 + 9 * (col_max - merged[col]) / (col_max - col_min)

# ── Calculate overall score = mean of normalized metrics ─────────────────
merged["Overall_Score"] = merged[normalized_col_names].mean(axis=1).round(2)
merged = merged.sort_values("Overall_Score", ascending=False).reset_index(drop=True)
merged["Rank"] = merged.index + 1

print("=== LINEAR LEADERBOARD (Normalized 1-10) ===")
print(merged[["Rank", "Ticker", "Category", "Overall_Score"]].to_string(index=False))

=== LINEAR LEADERBOARD (Normalized 1-10) ===
 Rank Ticker                     Category  Overall_Score
    1   NVDA            Individual Stocks           7.46
    2    GLD      Commodity ETFs (Metals)           7.13
    3    DBA Commodity ETFs (Agriculture)           6.93
    4    WMT            Individual Stocks           6.71
    5    SLV      Commodity ETFs (Metals)           6.31
    6   PPLT      Commodity ETFs (Metals)           6.20
    7   IBIT                  Crypto ETFs           6.01
    8    VTI            Broad Market ETFs           5.98
    9    XLU                  Sector ETFs           5.98
   10    JNJ            Individual Stocks           5.96
   11   SOYB Commodity ETFs (Agriculture)           5.87
   12   AAPL            Individual Stocks           5.56
   13   MSFT            Individual Stocks           5.32
   14    SPY            Broad Market ETFs           5.17
   15   WEAT Commodity ETFs (Agriculture)           5.13
   16   PALL      Commodity ETFs (Metals)  

In [14]:
# ── Success Metric Rankings: Top 3 Analysis ───────────────────────────────────

# Define the columns we want to rank (from radar_chart.ipynb)
metrics_to_rank = {
    "Downside Efficiency (Sortino)": "sortino__Normalized_Sortino_Score_1_10",
    "Drawdown Resilience (Calmar)": "calmar__Normalized_Calmar_Score_1_10",
    "Inflation Protection (Real Return)": "inflation__Normalized_Inflation_Score_1_10",
    "Market Independence (Correlation)": "market_independence__Correlation_Percentile",
    "Crisis Performance (Stress Test)": "crisis__Normalized_Crisis_Score_1_10"
}

print("=== INDIVIDUAL METRIC TOP 3 RANKINGS ===\n")

for label, col in metrics_to_rank.items():
    # Sort by the specific metric and take the top 3
    top_3 = merged.sort_values(col, ascending=False).head(3)
    
    print(f"--- {label} ---")
    print(top_3[["Ticker", "Category", col]].to_string(index=False))
    print("\n" + "="*40 + "\n")


=== INDIVIDUAL METRIC TOP 3 RANKINGS ===

--- Downside Efficiency (Sortino) ---
Ticker                Category  sortino__Normalized_Sortino_Score_1_10
  NVDA       Individual Stocks                                   10.00
   WMT       Individual Stocks                                    9.59
   GLD Commodity ETFs (Metals)                                    9.18


--- Drawdown Resilience (Calmar) ---
Ticker                Category  calmar__Normalized_Calmar_Score_1_10
   WMT       Individual Stocks                                 10.00
  NVDA       Individual Stocks                                  9.59
   GLD Commodity ETFs (Metals)                                  9.18


--- Inflation Protection (Real Return) ---
Ticker                Category  inflation__Normalized_Inflation_Score_1_10
  NVDA       Individual Stocks                                       10.00
   WMT       Individual Stocks                                        9.59
   SLV Commodity ETFs (Metals)                     

In [ ]:
def create_individual_radar(ticker_name, dataframe, save_output=False):
    """
    Creates and displays a radar chart for a specific ticker using 
    the team's standardized 1-10 normalized scores.
    """
    # 1. Configuration (Matching radar_chart.ipynb schema)
    categories = [
        "Sortino", 
        "Inflation", 
        "Calmar", 
        "Independence", 
        "Crisis"
    ]
    
    metric_cols = [
        "sortino__Normalized_Sortino_Score_1_10",
        "inflation__Normalized_Inflation_Score_1_10",
        "calmar__Normalized_Calmar_Score_1_10",
        "market_independence__Correlation_Percentile",
        "crisis__Normalized_Crisis_Score_1_10"
    ]

    # 2. Extract Data
    ticker_data = dataframe[dataframe['Ticker'] == ticker_name]
    
    if ticker_data.empty:
        print(f"Error: Ticker '{ticker_name}' not found in the dataset.")
        return

    # Extract values and close the radar loop
    values = ticker_data[metric_cols].values.flatten().tolist()
    values_closed = values + [values[0]]
    categories_closed = categories + [categories[0]]

    # 3. Create Figure
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=categories_closed,
        fill='toself',
        name=ticker_name,
        line=dict(color='#636EFA', width=3),
        fillcolor='rgba(99, 110, 250, 0.3)'
    ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 10], gridcolor="lightgrey"),
            angularaxis=dict(rotation=90, direction="clockwise")
        ),
        title=dict(
            text=f"<b>Performance Fingerprint: {ticker_name}</b>",
            x=0.5,
            font=dict(size=18)
        ),
        template="plotly_white",
        width=500,
        height=500
    )

    # 4. Save Logic
    # if save_output:
    #     output_dir = "../plots/individual_radars"
    #     os.makedirs(output_dir, exist_ok=True)
    #     fig.write_image(os.path.join(output_dir, f"radar_{ticker_name}.png") )
    #     print(f"✅ Saved radar_{ticker_name}.png to {output_dir}")

    fig.show()

In [19]:
target_tickers = [
    "GLD", 
    "WMT", 
    "DBA", 
    "NVDA", 
    "WEAT"
]

for i in target_tickers:
    create_individual_radar(i, merged)